In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from adjustText import adjust_text  # <-- 新增：用于防重叠标注的排版神器

# 忽略 log(0) 等计算警告
warnings.filterwarnings('ignore')

# --- 路径配置 ---
input_dir = r"C:\Users\28616\Desktop\DESeq2_Results"
output_root = r"C:\Users\28616\Desktop\Volcano_Plots_Cluster\onlyC"
os.makedirs(output_root, exist_ok=True)

# 筛选阈值
P_THRESHOLD = 0.05
LFC_THRESHOLD = 1.0  
TOP_N_LABELS = 5  # <-- 新增：你想在两边各标注几个最显著的点？

for n in range(6):
    file_name = f"OnlyC_Cluster{n}_DESeq2_Results.csv"
    file_path = os.path.join(input_dir, file_name)
    
    if not os.path.exists(file_path):
        print(f"Skipping: {file_name} not found.")
        continue
    
    # 1. 读取数据 (注意：加了 index_col=0，因为 DESeq2 的行名通常就是位点名)
    df = pd.read_csv(file_path, index_col=0)

    # 抽样防卡死
    df = df.sample(n=min(50000, len(df)), random_state=42)
    
    # 自动识别列名
    p_col = [c for c in df.columns if 'pvalue' in c.lower() or 'p.value' in c.lower()][0]
    lfc_col = [c for c in df.columns if 'log2foldchange' in c.lower() or 'lfc' in c.lower()][0]
    
    # 2. 坐标预处理
    plot_df = df.copy()
    plot_df['neg_log_p'] = -np.log10(plot_df[p_col].replace(0, 1e-300))
    
    # 3. 设置颜色分类
    plot_df['Group'] = 'Not Significant'
    plot_df.loc[(plot_df[p_col] < P_THRESHOLD) & (plot_df[lfc_col] > LFC_THRESHOLD), 'Group'] = 'Up-regulated'
    plot_df.loc[(plot_df[p_col] < P_THRESHOLD) & (plot_df[lfc_col] < -LFC_THRESHOLD), 'Group'] = 'Down-regulated'
    
    # --- 新增：挑选需要标注的 Top 位点 ---
    # 分别把上调和下调里，p值最小（即 neg_log_p 最大）的前 TOP_N_LABELS 个挑出来
    top_up = plot_df[plot_df['Group'] == 'Up-regulated'].nlargest(TOP_N_LABELS, 'neg_log_p')
    top_down = plot_df[plot_df['Group'] == 'Down-regulated'].nlargest(TOP_N_LABELS, 'neg_log_p')
    top_points = pd.concat([top_up, top_down])
    
    # 4. 绘图
    plt.figure(figsize=(9, 7)) # 稍微把画布拉大一点，给文字留空间
    sns.scatterplot(
        data=plot_df, x=lfc_col, y='neg_log_p', 
        hue='Group', 
        palette={'Not Significant': '#E0E0E0', 'Up-regulated': '#FF4B4B', 'Down-regulated': '#4B7BFF'},
        alpha=0.7, s=15, edgecolor=None # 稍微调大了点的大小 s=15
    )
    
    # --- 新增：把文字画上去 ---
    texts = []
    for idx, row in top_points.iterrows():
        # idx 就是位点的名字 (index)，如果你当时存的时候位点名在其他列，这里请改成 row['你的列名']
        t = plt.text(row[lfc_col], row['neg_log_p'], str(idx), fontsize=9, fontweight='bold')
        texts.append(t)
        
    # 召唤 adjustText 自动排版文字，避免遮挡散点，并加上小箭头
    if texts:
        adjust_text(texts, 
                    arrowprops=dict(arrowstyle='-', color='gray', lw=0.8), # 箭头样式
                    expand_points=(1.5, 1.5)) # 文字离点稍微远一点
    
    # 5. 添加辅助线
    plt.axhline(-np.log10(P_THRESHOLD), color='black', linestyle='--', linewidth=0.8, label=f'p-value < {P_THRESHOLD}')
    plt.axvline(LFC_THRESHOLD, color='gray', linestyle='--', linewidth=0.8, label=f'|log2FC| > {LFC_THRESHOLD}')
    plt.axvline(-LFC_THRESHOLD, color='gray', linestyle='--', linewidth=0.8)
    
    # 6. 图表装饰
    plt.title(f"Volcano Plot: Cluster {n} VS other Clusters", fontsize=15, pad=15)
    plt.xlabel("log2 Fold Change", fontsize=12)
    plt.ylabel("-log10(p-value)", fontsize=12)
    
    limit = max(abs(plot_df[lfc_col].min()), abs(plot_df[lfc_col].max())) * 1.1
    plt.xlim(-limit, limit)
    
    plt.legend(loc='upper right', fontsize='small')
    plt.grid(True, linestyle=':', alpha=0.4)
    
    # 7. 保存
    save_path = os.path.join(output_root, f"Cluster{n}_volcano_labeled_onlyC.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

print(f"✅ 绘图完成。带标注的火山图已保存至: {output_root}")

✅ 绘图完成。带标注的火山图已保存至: C:\Users\28616\Desktop\Volcano_Plots_Cluster\onlyC
